<a href="https://colab.research.google.com/github/yujass-DAlab/PFT-AI-Companion/blob/main/DA_PF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Portfolio Project: "Home Health Readmission Risk Analysis"

import sqlite3
import pandas as pd
import numpy as np

# Create in-memory database
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Create tables
cur.executescript("""
CREATE TABLE patients (
    patient_id TEXT PRIMARY KEY,
    age INTEGER,
    sex TEXT,
    enrollment_date TEXT,
    primary_diagnosis TEXT
);

CREATE TABLE visits (
    visit_id INTEGER PRIMARY KEY,
    patient_id TEXT,
    visit_date TEXT,
    visit_type TEXT,
    vital_status TEXT
);

CREATE TABLE labs (
    lab_id INTEGER PRIMARY KEY,
    patient_id TEXT,
    lab_date TEXT,
    test_name TEXT,
    result_value REAL,
    result_unit TEXT
);
""")

# Insert sample data
patients_data = [
    ("P1", 72, "F", "2026-01-01", "Heart Failure"),
    ("P2", 68, "M", "2026-01-15", "COPD"),
    ("P3", None, "F", "2026-02-01", "Diabetes"),   # missing age
    ("P4", 81, "M", "2026-02-10", "Heart Failure"),
    ("P5", 55, "F", "2026-03-01", "Hypertension"),
]

visits_data = [
    (1, "P1", "2026-01-05", "Home", "Stable"),
    (2, "P1", "2026-01-20", "ER", "Critical"),
    (3, "P1", "2026-02-10", "Home", "Stable"),
    (4, "P2", "2026-01-20", "Home", "Stable"),
    (5, "P2", "2026-02-15", "ER", "Critical"),
    (6, "P3", "2026-02-15", "Home", "Stable"),
    (7, "P4", "2026-02-20", "ER", "Critical"),
    (8, "P4", "2026-03-05", "Home", "Stable"),
    (9, "P5", "2026-03-10", "Home", "Stable"),
]

labs_data = [
    (1, "P1", "2026-01-04", "BNP", 450, "pg/mL"),
    (2, "P1", "2026-01-19", "BNP", 890, "pg/mL"),
    (3, "P2", "2026-01-19", "Creatinine", 1.4, "mg/dL"),
    (4, "P3", "2026-01-28", "Glucose", None, "mg/dL"),   # missing value
    (5, "P4", "2026-02-19", "BNP", 1200, "pg/mL"),
    (6, "P4", "2026-03-01", "Creatinine", 2.1, "mg/dL"),
    (7, "P5", "2026-03-09", "Glucose", 110, "mg/dL"),
    (8, "P2", "2026-01-10", "Creatinine", 1.2, "mg/dL"),  # lab before enrollment? check
]

cur.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients_data)
cur.executemany("INSERT INTO visits VALUES (?,?,?,?,?)", visits_data)
cur.executemany("INSERT INTO labs VALUES (?,?,?,?,?,?)", labs_data)
conn.commit()

# Helper function to run SQL and return a DataFrame
def run_sql(query):
    return pd.read_sql_query(query, conn)

print("✅ Database created with 5 patients, 9 visits, 8 labs.")

✅ Database created with 5 patients, 9 visits, 8 labs.


In [ ]:
"""Task 1 (SQL) — Lost to Follow‑Up
Write a SQL query that returns the total number of patients who have zero visits recorded.
Return a single number."""

result=run_sql("""
SELECT COUNT(*) AS total_patients
FROM patients p LEFT JOIN visits v ON p.patient_id=v.patient_id
WHERE v.patient_id IS NULL
""")
print(result)

   total_patients
0               0


In [ ]:
"""Task 2 (Pandas) — Age Imputation
In df_patients, Patient P3 has a missing age.
Write a Pandas script to fill this missing age with the median age of the other patients.
Update the original DataFrame."""

df_patients=run_sql("""
SELECT *
FROM patients
""")
df_patients['age']=df_patients['age'].fillna(df_patients['age'].median())

# OR

df_patients = pd.DataFrame(
    patients_data,
    columns=['patient_id', 'age', 'sex', 'enrollment_date', 'primary_diagnosis']
)
df_patients['age'] = df_patients['age'].fillna(df_patients['age'].median())
print(df_patients)

  patient_id   age sex enrollment_date primary_diagnosis
0         P1  72.0   F      2026-01-01     Heart Failure
1         P2  68.0   M      2026-01-15              COPD
2         P3  70.0   F      2026-02-01          Diabetes
3         P4  81.0   M      2026-02-10     Heart Failure
4         P5  55.0   F      2026-03-01      Hypertension


In [ ]:
"""Task 3 (SQL) — Most Recent Critical Visit
Using a CTE and ROW_NUMBER(), find the most recent ‘Critical’ visit for each patient.
Return patient_id, visit_date, vital_status."""

result=run_sql("""
WITH most_recent_critical AS (
SELECT patient_id, visit_date, vital_status, ROW_NUMBER () OVER (PARTITION BY patient_id ORDER BY visit_date DESC) AS rn
FROM visits
WHERE vital_status = 'Critical'
)
SELECT patient_id, visit_date, vital_status
FROM most_recent_critical
WHERE rn=1
""")

print(result)

  patient_id  visit_date vital_status
0         P1  2026-01-20     Critical
1         P2  2026-02-15     Critical
2         P4  2026-02-20     Critical


In [ ]:
"""Task 4 (Pandas) — Lab Cleanup
In df_labs, drop any rows where result_value is missing. Save as df_labs_clean."""

df_labs=pd.read_sql_query("SELECT * FROM labs", conn)
df_labs_clean=df_labs.dropna(subset='result_value')
print(df_labs_clean)

   lab_id patient_id    lab_date   test_name  result_value result_unit
0       1         P1  2026-01-04         BNP         450.0       pg/mL
1       2         P1  2026-01-19         BNP         890.0       pg/mL
2       3         P2  2026-01-19  Creatinine           1.4       mg/dL
4       5         P4  2026-02-19         BNP        1200.0       pg/mL
5       6         P4  2026-03-01  Creatinine           2.1       mg/dL
6       7         P5  2026-03-09     Glucose         110.0       mg/dL
7       8         P2  2026-01-10  Creatinine           1.2       mg/dL


In [ ]:
"""Task 5 (Clinical Logic) — Impossible Timeline
Join patients and labs. Identify any patient who had a lab drawn before their enrollment date.
In 2–3 sentences, state your immediate next step as a data analyst."""

# SQL Join patients and labs to identify patints:
result=run_sql("""
SELECT p.patient_id
FROM patients p JOIN labs l On p.patient_id=l.patient_id
WHERE julianday(l.lab_date) < julianday(p.enrollment_date)
""")
print(result)

#Immediate Next step as a Data Analyst:
# Having lab date and value before the enrollment date usually violates trial protocols. I need to review the current trial protocol and confirm if this is allowed.
# If not, I need to flag this as logical error and data discrepancy, issue a data query to the source and document the SDV for the two abnormal values.

  patient_id
0         P3
1         P2


In [ ]:
"""Task 6 (MLOps) — Silent Failure
A production model was trained on result_unit = 'mg/dL'. Now the API sends mmol/L.
The model doesn’t crash but predictions are nonsensical."""

# Name the phenomenon:
# The phenomenon is called schema mismatching and it is a type of Training-Serving-Skew in the model pipeline.

# Describe one architectural fix:
# Implement strict Schema Validation at the API Gateway (or FastAPI endpoint).
# If the incoming JSON payload contains an unexpected unit (e.g., 'mmol/L' instead of 'mg/dL'),
# the API should immediately reject the request with a 400 Bad Request error.
# This ensures bad data never reaches the model, turning a silent failure into a loud, catchable error.

In [ ]:
"""Task 7 (System Design) — Unreliable Network
A rural clinic has frequent internet outages. They need the model to run locally."""

# Name the deployment strategy:
# Edge Deployment is what needed here.

# Mention one tool or technique that enables it:
# Two of the many tools to enable models wihtout reliable internet are ONNX Runtime and TensorFlow Lite for on‑device inference.

In [ ]:
"""Task 8 (SQL + Pandas) — Readmission Rate
Assume any patient with at least one ‘Critical’ visit is considered “readmitted.”"""

# SQL: Write a query that returns the percentage of patients who had at least one critical visit.
result=run_sql("""
SELECT COUNT(DISTINCT CASE WHEN vital_status ='Critical' THEN patient_id END) *100.0 / NULLIF(COUNT(DISTINCT patient_id), 0) AS readmitted_percent
FROM visits
""")
print(result)

# Pandas: Write the equivalent code.
visits_df=pd.read_sql_query("SELECT * FROM visits", conn)
readmitted=visits_df[visits_df['vital_status'] =='Critical']['patient_id'].nunique()
eligible=visits_df['patient_id'].nunique()
if eligible > 0:
    readmitted_percentage = (readmitted * 100.0) / eligible
else:
    readmitted_percentage = None
print(readmitted_percentage)

   readmitted_percent
0                60.0
60.0


In [ ]:
"""Step 3: Deliverables (for your portfolio)
Submit the following in a single notebook or markdown file:
All 8 tasks with code and outputs.
A 1‑page executive summary answering: What is the readmission rate in this cohort? Which patients are highest risk?
What data quality issues did you find? What would you recommend to leadership?
A short reflection (3–5 sentences):
What was the hardest task?
What did you learn about patient‑level vs row‑level analysis?"""

"""
1. Executive Summary
Project Overview:
This analysis evaluated a cohort of 5 home health patients to determine 30-day readmission risk, identify high-risk patient profiles, and assess the quality of the underlying clinical data pipeline.

Key Findings:
Readmission Rate: The readmission rate for this cohort is 60% (3 out of 5 patients had at least one 'Critical' visit).
                  (Note: For this project, "readmission" is defined as any patient who had at least one 'Critical' visit, serving as a proxy for an unplanned ER visit or hospital readmission)

Highest-Risk Patients: Patients P1 (72, Heart Failure) and P4 (81, Heart Failure) represent the highest risk.
Both are elderly patients with Heart Failure who experienced 'Critical' visits (ER visits) and had severe lab abnormalities (e.g., P1's BNP spiked to 890 pg/mL, and P4's BNP was 1200 pg/mL).

Data Quality Issues: The data audit revealed three critical issues:
                     Missing clinical data (Patient P3 had no recorded age, and one Glucose lab result was missing a value).
                     Impossible timelines: Patient P2 had a lab drawn on 2026-01-10, which is 5 days before their official enrollment date (2026-01-15).
                     Missing follow-up (none in this specific dataset, but the query logic was established).

Recommendations for Leadership:
Clinical Intervention: Implement a targeted remote-monitoring protocol for elderly patients (65+) with a primary diagnosis of Heart Failure to catch BNP spikes before they require an ER visit.

Data Validation: Implement automated schema validation at the EDC (Electronic Data Capture) level.
                 The system should also prevent sites from entering lab dates that occur before a patient's enrollment date.
                 Missing ages should also trigger an immediate Data Query to the site.

2. Short Reflection
The hardest task in this project was Task 8 (Readmission Rate), which required calculating the percentage of patients with critical visits.
The challenge was not the clinical logic, but the syntax: I frequently found myself mixing up commands and data structures between SQL and Pandas.
For example, remembering to use COUNT(DISTINCT CASE WHEN...) in SQL versus .nunique() in Pandas required careful attention to the specific language requirements.

This project also highlighted the critical difference between row-level and patient-level analysis.
Row-level analysis treats each record as an isolated event, which is useful for tracking individual labs or visits.
However, patient-level analysis is essential for clinical outcomes because it accounts for the patient's identity.
As I saw with this dataset, a single patient can have multiple encounters (like P1, who had both Home and ER visits).
Grouping and aggregating by patient_id ensures that a patient is only counted once in the denominator, preventing the readmission rate from being skewed by patients who visit the hospital frequently."""
